In [1]:
import gradio as gr
import os
from dotenv import load_dotenv
!ollama pull llama3.2
load_dotenv(override=True)

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕████████

True

In [2]:

from openai import OpenAI, Stream
openai=OpenAI()
ollama=OpenAI(base_url="http://localhost:11434/v1",api_key="olama")
system_prompt=""" you are a very good AI assistant. you will give me proper answer of my question.
"""
def gpt_chat(prmpt):
    
    user_prompt=prmpt
    stream=openai.chat.completions.create(model="gpt-4.1-mini", messages=[{"role":"system","content": system_prompt},{"role":"user","content":user_prompt}],stream=True)
    result=" "
    for chunk in stream:
        result +=chunk.choices[0].delta.content or ""
        yield result
    
def ollama_chat(prmpt):

    user_prompt=prmpt
    stream=ollama.chat.completions.create(model="llama3.2", messages=[{"role":"system","content": system_prompt},{"role":"user","content":user_prompt}],stream=True)
    result=" "
    for chunk in stream:
        result +=chunk.choices[0].delta.content or ""
        yield result

In [3]:



def chat_bot(prompt,value):
    yield " "
    if value =="gpt":
        result = gpt_chat(prompt)
    elif value =="ollama":
        result = ollama_chat(prompt)
    else:
        print("value error")
    yield from result


In [4]:

message_input = gr.Textbox(label="Your question :", info="It will be single shot promt", lines=7)
message_output = gr.Markdown(label="Response:")
message_choices= gr.Dropdown(["gpt", "ollama"], info="Choice")
GR_Interface=gr.Interface(fn=chat_bot,title="Kasoul bot", inputs=[message_input,message_choices],outputs=[message_output], examples=[
            ["Explain the Transformer architecture to a layperson", "gpt"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "ollama"]
        ],flagging_mode="never")
GR_Interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:

system_prompt=""" you are a very good AI assistant.You will be given a website link, you will analyse it and give me proper summary of that webpage"""

def gpt_chat(prmpt):
    
    user_prompt=prmpt
    stream=openai.chat.completions.create(model="gpt-4.1-mini", messages=[{"role":"system","content": system_prompt},{"role":"user","content":user_prompt}],stream=True)
    result=" "
    for chunk in stream:
        result +=chunk.choices[0].delta.content or ""
        yield result
    
def ollama_chat(prmpt):

    user_prompt=prmpt
    stream=ollama.chat.completions.create(model="llama3.2", messages=[{"role":"system","content": system_prompt},{"role":"user","content":user_prompt}],stream=True)
    result=" "
    for chunk in stream:
        result +=chunk.choices[0].delta.content or ""
        yield result

In [6]:
from scraper import fetch_website_contents
def chat_bot_website(name,value,id):
    user_Prompt=f""" Please generate the Broucher of {name} this webpage, here is the front page:\n\n"""
    user_Prompt+=fetch_website_contents(value)
    yield " "
    if id =="gpt":
        result = gpt_chat(value)
    elif id =="ollama":
        result = ollama_chat(value)
    else:
        print("value error")
    yield from result


In [9]:
message_input = gr.Textbox(label="Your website name :", info="It will be single shot promt", lines=7)
message_link=gr.Textbox(label="Your website link:", info="It will fetch and give you the summary", lines=7)
message_output = gr.Markdown(label="Response:")
message_choices= gr.Dropdown(["gpt", "ollama"], info="Choice")
GR_Interface=gr.Interface(
    fn=chat_bot_website,
title="Kasoul bot", 
inputs=[message_input,message_link,message_choices],
outputs=[message_output], 
examples=[
            ["Hugging Face", "https://huggingface.co", "gpt"],
            ["Edward Donner", "https://edwarddonner.com", "ollama"]
        ],
        flagging_mode="never")
GR_Interface.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


### here i will create a callback function which will be called by gradio for each input(user prompt) will pass and the it will collect user input and history with it.



In [11]:
def callback(messages, history):
    return f"message is : {messages} \n and history is {history}"


In [15]:


gr.ChatInterface(fn=callback,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [27]:
system_prompt = "You are a helpful assistant"
def callback(messages, history):
    history=[{"role":h["role"],"content":h["content"]} for h in history]

    #return f"message is : {messages} \n and history is {history}"
    message=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":messages}]
    stream=openai.chat.completions.create(model="gpt-4.1-mini",messages=message,stream=True)
    response=""
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ""
        yield response


    
    

In [28]:



gr.ChatInterface(fn=callback,type="messages", title="Kasoul BOT").launch(auth=["admin","admin"],inbrowser=True,share=True)

* Running on local URL:  http://127.0.0.1:7886
* Running on public URL: https://23e5d3eee3c09fa9c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
